# Generate Monolayer Disk Summary

This notebook extracts the monolayer disk diameter and cell count for each PhysiCell snapshot, writes the time series to CSV, and shows the final SVG snapshot.

In [1]:
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from scipy.io import loadmat
from IPython.display import SVG, display

In [2]:
ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'PhysiCell').is_dir()),
    Path('/home/tntiniak/Work/observatory_benchmark'),
)
output_folder = ROOT / 'PhysiCell' / 'results' / 'monolayer_28'
snapshot_xml_files = sorted(output_folder.glob('output*.xml'))
output_csv = output_folder / 'monolayer_disk_summary.csv'

if not snapshot_xml_files:
    raise FileNotFoundError(f'No output XML files found in {output_folder}')

output_folder, len(snapshot_xml_files), output_csv

(PosixPath('/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/monolayer_28'),
 673,
 PosixPath('/home/tntiniak/Work/observatory_benchmark/PhysiCell/results/monolayer_28/monolayer_disk_summary.csv'))

In [3]:
def compute_disk_diameter(points, radii):
    if len(points) == 0:
        return 0.0

    center = points.mean(axis=0)
    radial_extent = np.linalg.norm(points - center, axis=1) + radii
    return float(2.0 * radial_extent.max())

results = []

for xml_path in snapshot_xml_files:
    root = ET.parse(xml_path).getroot()
    current_time = float(root.findtext('.//current_time'))
    mat_name = root.findtext('.//simplified_data/filename')
    if mat_name is None:
        raise ValueError(f'No cell data filename found in {xml_path}')

    cells = loadmat(output_folder / mat_name)['cells']
    cell_data = cells[[0, 1, 2, 3, 37]]
    df_mat = pd.DataFrame(cell_data.T, columns=['id', 'x', 'y', 'z', 'radius'])

    points = df_mat[['x', 'y']].to_numpy(dtype=float)
    radii = df_mat['radius'].to_numpy(dtype=float)
    cell_count = int(len(df_mat))
    diameter = compute_disk_diameter(points, radii)

    results.append({
        'timestep': current_time,
        'cell_count': cell_count,
        'diameter': diameter,
    })

df_summary = pd.DataFrame(results).sort_values('timestep').reset_index(drop=True)
df_summary.to_csv(output_csv, index=False)
df_summary.tail()

,timestep,cell_count,diameter
668,40079.999999,26920,1725.348770
669,40139.999999,27050,1728.193034
670,40199.999999,27168,1730.233174
671,40259.999999,27248,1730.684111
672,40319.999999,27337,1731.909914


In [4]:
final_svg_path = output_folder / 'final.svg'
if not final_svg_path.exists():
    raise FileNotFoundError(f'No final SVG snapshot found at {final_svg_path}')

print(f"Saved CSV to {output_csv}")
print(f"Final snapshot: timestep={df_summary.iloc[-1]['timestep']:.0f} min | cells={int(df_summary.iloc[-1]['cell_count'])} | diameter={df_summary.iloc[-1]['diameter']:.2f} microns")
display(SVG(filename=str(final_svg_path)))

Saved CSV to /home/tntiniak/Work/observatory_benchmark/PhysiCell/results/monolayer_28/monolayer_disk_summary.csv
Final snapshot: timestep=40320 min | cells=27337 | diameter=1731.91 microns
